<h1 style="color:green;font-size:22px;">Function 7 - Black-Box Optimisation</h1>
<h1 style="color:#0000CD;font-size:19px;"">Introduction and illustrative analogy</h1>

**Function 7** is a six-dimensional black-box objective defined over the bounded domain $[0,1]^6$. Its analytical form and physical interpretation are unknown. The objective is to identify high-value input configurations under a limited sequential-query budget. The goal is to maximize **Function 7**.

An illustrative analogy is the tuning of six machine-learning hyperparameters, where each input vector represents a candidate configuration and the returned objective value represents an unknown performance score. This analogy is conceptual only: no semantic interpretation of the six coordinates is assumed in the optimisation.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as s
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

from itertools import combinations
from mpl_toolkits.mplot3d import Axes3D
from sklearn.exceptions import ConvergenceWarning

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel
warnings.filterwarnings("ignore", category=ConvergenceWarning)

<h1 style="color:#0000CD;font-size:19px;"">Week 1</h1>

**1.1 - Extraction of Initial Data**

In [2]:
inputs = np.load('Initial Data/function_7/initial_inputs.npy')
outputs = np.load('Initial Data/function_7/initial_outputs.npy')
print(inputs.shape, outputs.shape)

(30, 6) (30,)


In [3]:
data = pd.DataFrame(inputs, columns=['x1','x2','x3','x4','x5','x6'])
data['y'] = outputs
display(data)

,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


In [4]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

1.3649683044991994 1.362266839474691


**1.2 - Optimisation (GP + UCB)**

With only 30 initial observations in six dimensions, the global modality of the objective cannot be established reliably. A Gaussian Process surrogate with an RBF kernel is therefore combined with Upper Confidence Bound acquisition to balance predicted performance and model uncertainty.

In [5]:
# 6D Cartesian Candidate grid
x1 = np.linspace(0,1,15)
x2 = np.linspace(0,1,15)
x3 = np.linspace(0,1,15)
x4 = np.linspace(0,1,15)
x5 = np.linspace(0,1,15)
x6 = np.linspace(0,1,15)

xx1, xx2, xx3, xx4, xx5, xx6 = np.meshgrid(x1, x2, x3, x4, x5, x6)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel(), xx3.ravel(), xx4.ravel(), xx5.ravel(), xx6.ravel()])
del xx1, xx2, xx3, xx4, xx5, xx6

# Predict GP mean and uncertainty
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound
kappa = 1.5
ucb = y_pred + kappa * sigma
index_max = np.argmax(ucb)
next_point = np.array(X_grid[index_max], float)
print(f"Next point (with GP+UCB):{next_point[0]:.6f}-{next_point[1]:.6f}-{next_point[2]:.6f}-{next_point[3]:.6f}-{next_point[4]:.6f}-{next_point[5]:.6f}")


Next point (with GP+UCB):0.071429-0.428571-0.285714-0.142857-0.357143-0.714286


<h1 style="color:#0000CD;font-size:19px;"">Week 2</h1>

**2.1 - Previous Week's Query Result**

In [6]:
# New Query Point
x_new = np.array([[0.071429, 0.428571, 0.285714, 0.142857, 0.357143, 0.714286]])
y_new = 1.7016886260587805

def add_QueriedPoint(data, x_new, y_new):

    inputs = data[['x1', 'x2','x3', 'x4','x5','x6']].to_numpy()
    outputs = data['y'].to_numpy()
    
    # Checks if New Points is already included in data
    exists = False
    for i in range(inputs.shape[0]):
        if np.allclose(inputs[i], x_new) and np.isclose(outputs[i], y_new):
            exists = True
            break

    # Only adds if it doesn't exist already 
    if not exists:
        inputs = np.vstack([inputs, x_new])
        outputs = np.append(outputs, y_new)
        data = pd.DataFrame(inputs, columns=['x1', 'x2','x3', 'x4','x5','x6']).assign(y=outputs)
    
        print("Point added!")
    else:
        print("Point already exists, skipping addition.")
    return data

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


In [7]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

1.7016886260587805 1.6989871610342722


**2.2 - Next Points Queried**

In [8]:
## ReUse of the 6D Cartesian Grid X_grid

# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 1.5
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.         0.35714286 0.35714286 0.         0.28571429 0.78571429]


<h1 style="color:#0000CD;font-size:19px;"">Week 3</h1>

**3.1 - Previous Week's Query Result**

In [9]:
# New Query Point
x_new = np.array([[0.00, 0.357143, 0.357143, 0.00, 0.285714, 0.785714]])
y_new = 1.0177367185247268

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


In [10]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

1.7016886260587805 1.6989871610342722


**3.2 - Next Points Queried**

In [11]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.07142857 0.35714286 0.28571429 0.21428571 0.28571429 0.64285714]


<h1 style="color:#0000CD;font-size:19px;"">Week 4</h1>

**4.1 - Previous Week's Query Result**

In [12]:
# New Query Point
x_new = np.array([[0.071429, 0.357143, 0.285714, 0.214286, 0.285714, 0.642857]])
y_new = 2.329547987458463

# Checks if New Points is already included in data
data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


In [13]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

2.329547987458463 2.3268465224339545


**4.2 - Next Point Query Selection: UCB with Cartesian-Grid**

In [14]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("For comparison purposes only")
next_point = next_point_ucb
print(f"Next point (UCB + Cartesian Grid): {next_point}")

For comparison purposes only
Next point (UCB + Cartesian Grid): [0.07142857 0.28571429 0.21428571 0.21428571 0.28571429 0.5       ]


**Observation:** Successive grid-based UCB proposals are concentrated in a similar region. To improve candidate-space coverage without increasing the size of the full tensor grid, the next iteration makes two changes:

- replace the Cartesian-grid candidates with a scrambled Sobol sequence;
- replace the RBF kernel with a Matérn kernel using $\nu=2.5$.

The acquisition rule remains UCB for this iteration.

**4.3 - Next Point Alternative Query Selection: UCB with Sobol Candidates**

In [15]:
from scipy.stats import qmc

d = inputs.shape[1]
m = 13
sampler = qmc.Sobol(d=d, scramble=True, seed=42)
X_cand = sampler.random_base2(m)

# GP Fit First
kernel = Matern(length_scale=[0.2]*d, nu=2.5) + WhiteKernel(noise_level=1e-5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_cand, return_std=True)


# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_cand[index_ucb]

# Continue exploration using UCB
print("Next point submitted")
next_point = next_point_ucb
print(f"Next point (UCB+ Sobol): {next_point}")

Next point submitted
Next point (UCB+ Sobol): [0.04405559 0.1349121  0.1231442  0.13303968 0.26777369 0.55495323]


<h1 style="color:#0000CD;font-size:19px;"">Week 5</h1>

**5.1 - Previous Week's Query Result**

In [16]:
# New Query Point
x_new = np.array([[0.044056, 0.134912, 0.123144, 0.133040, 0.267774, 0.554953]])
y_new = 1.7347585659642555

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


In [17]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

2.329547987458463 2.3268465224339545


**5.2 - Next Query**

The global Sobol query returned y=1.734759 and did not improve the incumbent value of 2.329548. Nevertheless, the preceding optimisation rounds had identified a coherent high-value region around the incumbent. The search therefore moved from global exploration to local refinement rather than continuing unrestricted global sampling.

The surrogate is upgraded to a Gaussian Process with a scaled Matérn-$5/2$ kernel and a small white-noise component. Expected Improvement is used as the primary query-selection rule, while UCB is retained as a diagnostic comparison.

In [18]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(5)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("Transition to EI Refinement - UCB agrees")

Current best point: [0.071429 0.357143 0.285714 0.214286 0.285714 0.642857]
Current best value: 2.329547987458463
Next point (EI): [0.01228348 0.25987178 0.37329352 0.21526748 0.3086249  0.63382556]
Next point (UCB): [0.01228348 0.25987178 0.37329352 0.21526748 0.3086249  0.63382556]
Transition to EI Refinement - UCB agrees


<h1 style="color:#0000CD;font-size:19px;"">Week 6</h1>

**6.1 - Previous Week's Query Result**

In [19]:
# New Query Point
x_new = np.array([[0.012283, 0.259872, 0.373294, 0.215267, 0.308625, 0.633826]])
y_new = 2.580636316989488

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


**6.2 - Next Query**

In [20]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14
sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(6)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI continuation for next candidate")

Current best point: [0.012283 0.259872 0.373294 0.215267 0.308625 0.633826]
Current best value: 2.580636316989488
Next point (EI): [0.00190272 0.18372043 0.41863036 0.23498747 0.30526335 0.6236302 ]
Next point (UCB): [0.00394439 0.24794883 0.47160642 0.2646952  0.30788885 0.58412744]
EI continuation for next candidate


From now on, the local optimisation policy is re-fitted after every returned observation. Candidate points are generated using the same scrambled Sobol design within a radius-0.12 box centred on the current incumbent. EI with ξ=0.005 is the query-selection rule, while UCB with $κ$=2.5/
sqrt(t) is retained as a diagnostic comparison. The decreasing UCB coefficient progressively reduces the diagnostic emphasis on uncertainty as the query budget is consumed.


<h1 style="color:#0000CD;font-size:19px;"">Week 7</h1>

**7.1 - Previous Week's Query Result**

In [21]:
# New Query Point
x_new = np.array([[0.001903, 0.183720, 0.418630, 0.234987, 0.305263, 0.623630]])
y_new = 2.7368110863855195

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


**7.2 - Next Query**

In [22]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(7)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI next candidate selected")

Current best point: [0.001903 0.18372  0.41863  0.234987 0.305263 0.62363 ]
Current best value: 2.7368110863855195
Next point (EI): [0.00363488 0.17179683 0.51694242 0.2844152  0.30452685 0.57393144]
Next point (UCB): [0.000799   0.06810944 0.37871066 0.29356129 0.29888088 0.74240612]
EI next candidate selected


<h1 style="color:#0000CD;font-size:19px;"">Week 8</h1>

**8.1 - Previous Week's Query Result**

In [23]:
# New Query Point
x_new = np.array([[0.003635, 0.171797, 0.516942, 0.284415, 0.304527, 0.573931]])
y_new = 2.6846004725291777

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


**8.2 - Next Query**

In [24]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(8)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI next candidate selected - matches with UCB")

Current best point: [0.001903 0.18372  0.41863  0.234987 0.305263 0.62363 ]
Current best value: 2.7368110863855195
Next point (EI): [0.00175342 0.10756843 0.46396636 0.25470747 0.30190135 0.6134342 ]
Next point (UCB): [0.00175342 0.10756843 0.46396636 0.25470747 0.30190135 0.6134342 ]
EI next candidate selected - matches with UCB


<h1 style="color:#0000CD;font-size:19px;"">Week 9</h1>

**9.1 - Previous Week's Query Result**

In [25]:
# New Query Point
x_new = np.array([[0.001753, 0.107568, 0.463966, 0.254707, 0.301901, 0.613434]])
y_new = 2.7727071973235726

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


**9.2 - Next Query**

In [26]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(9)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI next candidate selected - UCB matches with EI")

Current best point: [0.001753 0.107568 0.463966 0.254707 0.301901 0.613434]
Current best value: 2.7727071973235726
Next point (EI): [0.00339898 0.02712195 0.55829297 0.25243146 0.30668387 0.54131094]
Next point (UCB): [0.00339898 0.02712195 0.55829297 0.25243146 0.30668387 0.54131094]
EI next candidate selected - UCB matches with EI


<h1 style="color:#0000CD;font-size:19px;"">Week 10</h1>

**10.1 - Previous Week's Query Result**

In [27]:
# New Query Point
x_new = np.array([[0.003399, 0.027122, 0.558293, 0.252431, 0.306684, 0.541311]])
y_new = 2.3841696254643674

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


**10.2 - Next Query**

In [28]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(10)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI next candidate selected - matches UCB")

Current best point: [0.001753 0.107568 0.463966 0.254707 0.301901 0.613434]
Current best value: 2.7727071973235726
Next point (EI): [0.00352076 0.13548575 0.4889382  0.2736572  0.30351399 0.68535939]
Next point (UCB): [0.00352076 0.13548575 0.4889382  0.2736572  0.30351399 0.68535939]
EI next candidate selected - matches UCB


<h1 style="color:#0000CD;font-size:19px;"">Week 11</h1>

**11.1 - Previous Week's Query Result**

In [29]:
# New Query Point
x_new = np.array([[0.003521, 0.135486, 0.488938, 0.273657, 0.303514, 0.685359]])
y_new = 2.832769152582964

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


**11.2 - Next Query**

In [30]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(11)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI next candidate selected")

Current best point: [0.003521 0.135486 0.488938 0.273657 0.303514 0.685359]
Current best value: 2.832769152582964
Next point (EI): [0.00100993 0.07001164 0.59839225 0.29135286 0.33142847 0.65862158]
Next point (UCB): [0.01758758 0.14648897 0.40132833 0.28327118 0.29664123 0.64287192]
EI next candidate selected


<h1 style="color:#0000CD;font-size:19px;"">Week 12</h1>

**12.1 - Previous Week's Query Result**

In [31]:
# New Query Point
x_new = np.array([[0.001010, 0.070012, 0.598392, 0.291353, 0.331428, 0.658622]])
y_new = 2.625874476563031

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


**12.2 - Next Query**

In [32]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(12)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI next candidate selected")

Current best point: [0.003521 0.135486 0.488938 0.273657 0.303514 0.685359]
Current best value: 2.832769152582964
Next point (EI): [0.00686185 0.13819694 0.47081683 0.29563919 0.26762458 0.64813146]
Next point (UCB): [0.02431213 0.15468029 0.56341833 0.28775912 0.28153423 0.66951727]
EI next candidate selected


<h1 style="color:#0000CD;font-size:19px;"">Week 13</h1>

**13.1 - Previous Week's Query Result**

In [33]:
# New Query Point
x_new = np.array([[0.006862, 0.138197, 0.470817, 0.295639, 0.267625, 0.648131]])
y_new = 2.7739917003909733

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


**13.2 - Next Query**

With one query remaining, the EI offset is reduced from ξ=0.005 to ξ=0, prioritising expected improvement around the incumbent over additional exploratory displacement.

In [34]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.00
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# 5B. UCB Acquisition Query

kappa = 2.5 / np.sqrt(13)   # more exploitative than 3/sqrt(5)
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI next candidate selected - UCB matches")

Current best point: [0.003521 0.135486 0.488938 0.273657 0.303514 0.685359]
Current best value: 2.832769152582964
Next point (EI): [0.01758758 0.14648897 0.40132833 0.28327118 0.29664123 0.64287192]
Next point (UCB): [0.01758758 0.14648897 0.40132833 0.28327118 0.29664123 0.64287192]
EI next candidate selected - UCB matches


<h1 style="color:#0000CD;font-size:19px;"">14. Final Result</h1>

**14.1 -  Previous Week's Query Result**

In [35]:
# Queried Point result
x_new = np.array([[0.017588, 0.146489, 0.401328, 0.283271, 0.296641, 0.642872]])
y_new = 2.8546263939670093

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2', 'x3', 'x4', 'x5', 'x6']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


- The sequential Bayesian optimisation process improved the best observed objective value from 1.364968 in the initial sample to 2.854626 after 13 submitted queries, representing an absolute improvement of 1.489658 and a relative increase of approximately 109.1%.
- The search progressed from broad UCB-driven exploration toward local EI-based refinement as a promising region emerged, with several non-improving queries contributing information about the surrounding response surface. The final query produced the overall incumbent at ([0.017588,\ 0.146489,\ 0.401328,\ 0.283271,\ 0.296641,\ 0.642872]).
- Although the hidden nature of the objective prevents any claim that this point is the global optimum, the observed progression demonstrates that the adaptive strategy successfully identified and refined a substantially higher-value region within the available query budget.